In [ ]:
# v2 HANGAR Paper Trading Algorithm
# ==================================
# CELL 1: Install packages
# CELL 2: Imports
# CELL 3: Load .env (API keys)
# CELL 4: Run live paper trading
# CELL 5: View performance
# CELL 6: Risk-adjusted metrics (guarded, min 20 obs)
# CELL 7: Live stat tests (index alignment fixed)
# CELL 8: 5-year backtest (same universe, with transaction costs)

In [ ]:
%pip install --upgrade alpaca-trade-api pandas numpy matplotlib scipy yfinance python-dotenv
print("All required packages are installed.")

In [ ]:
import os
import sys

# Ensure hangar package is importable
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from hangar.v1_paper_trading import (
    V1_UNIVERSE,
    AlpacaPaperTrading,
    V1StrategyEngine,
    compute_risk_metrics,
    get_v1_performance_history,
    run_v1_5year_backtest,
    run_v1_live_paper_trading,
    run_v1_live_stat_tests,
    v1_live_account_progress,
    validate_universe,
)
from hangar.backtest.metrics import (
    sharpe_ratio,
    sortino_ratio,
    max_drawdown,
    calmar_ratio,
    win_rate,
)

print("\u2713 All modules imported successfully")

In [ ]:
# Load API keys from .env file (never hardcode keys!)
from dotenv import load_dotenv
load_dotenv(os.path.join(repo_root, '.env'))

api_key = os.getenv('ALPACA_API_KEY')
secret_key = os.getenv('ALPACA_SECRET_KEY')

if not api_key or not secret_key:
    raise ValueError(
        "ALPACA_API_KEY and ALPACA_SECRET_KEY must be set.\n"
        "Copy .env.example to .env and fill in your keys."
    )

print("\u2713 API keys loaded from .env")

In [ ]:
# Run v1 live paper trading
# - Universe is validated (deduped + tradability checked)
# - Value signal uses earnings yield with inverse-vol fallback
alpaca = run_v1_live_paper_trading(
    api_key=api_key,
    secret_key=secret_key,
    tickers=V1_UNIVERSE,
)

In [ ]:
# View 30-day performance history
perf = get_v1_performance_history(
    api_key=api_key,
    secret_key=secret_key,
)

In [ ]:
# Live account progress + risk-adjusted metrics
# Guarded: prints warning if < 20 observations available
live_metrics, live_df = v1_live_account_progress(
    api_key=api_key,
    secret_key=secret_key,
    period='3M',
    timeframe='1D',
    risk_free_rate=0.045,
    min_obs=20,
)

In [ ]:
# Live stat significance tests (Strategy vs SPY)
# Fix: index alignment normalised to date objects
live_stats = run_v1_live_stat_tests(
    api_key=api_key,
    secret_key=secret_key,
    period='3M',
    timeframe='1D',
    risk_free_rate=0.045,
    benchmark='SPY',
    min_obs=20,
)

In [ ]:
# 5-year backtest using the SAME universe as live trading
# Includes 10 bps transaction cost per rebalance turnover
backtest_results = run_v1_5year_backtest(
    tickers=V1_UNIVERSE,
    initial_capital=100_000,
    risk_free_rate=0.045,
    cost_bps=10,
)